# Bussin -- CPU ETL worker

All data work happens here, **not** in a GPU session.

The GPU quota is a *GPU* quota: CPU-only sessions do not touch it, and Kaggle allows 5 concurrent batch CPU sessions of 12 h each. That is roughly 60 free CPU-hours per wave for mining, cleaning, deduplication, tokenization and sharding.

Set `STAGE` and `SHARD_INDEX`, then Save & Run All. Launch five copies with `SHARD_INDEX` 0..4 to fan out.

**Accelerator must be None.**


In [ ]:
# --- Bussin worker setup ---------------------------------------------
# Pinned and quiet: every second here is a second of quota not spent on
# matrix multiplies. Target is >= 95% of session wall time in training steps.
import os, subprocess, sys, time
T0 = time.time()

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"      # parallel checkpoint pulls
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not os.path.exists("bussin"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/CHANGEME/bussin.git", "."], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "hf_transfer", "safetensors", "huggingface_hub", "pyyaml", "regex"],
               check=False)

sys.path.insert(0, os.getcwd())
print(f"setup took {time.time() - T0:.1f}s")

In [ ]:
# --- Credentials ------------------------------------------------------
# Store the token in Kaggle "Add-ons -> Secrets" as HF_TOKEN. Never paste a
# token into a notebook cell: notebooks get shared, and the token grants write
# access to your checkpoint repo.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle secrets")
except Exception as exc:
    print(f"could not load Kaggle secret ({exc}); falling back to env HF_TOKEN")
    assert os.environ.get("HF_TOKEN"), "HF_TOKEN is required"

In [ ]:
# --- Data pipeline (CPU only) -----------------------------------------
# CPU sessions do not consume the GPU quota, and Kaggle allows 5 concurrent
# batch CPU sessions of 12 h each. That is ~60 CPU-hours per wave, free, and it
# is why no data work should ever happen inside a GPU session.
import subprocess, sys

STAGE = "lexicon"       # lexicon | discover | mine | tokenize | shard
SHARD_INDEX = 0          # 0..4, so five sessions split the work
N_SHARDS = 5

cmds = {
    "lexicon":  [sys.executable, "pipelines/00_build_lexicon.py",
                 "--out", "/kaggle/working/lexicon/lexicon.jsonl"],
    "discover": [sys.executable, "pipelines/01_discover_emerging.py",
                 "--lexicon", "/kaggle/input/bussin-lexicon/lexicon.jsonl",
                 "--out", "/kaggle/working/lexicon/lexicon.jsonl",
                 "--recent-rows", "0"],
    "mine":     [sys.executable, "pipelines/02_mine_corpus.py",
                 "--shard-index", str(SHARD_INDEX), "--n-shards", str(N_SHARDS),
                 "--out", "/kaggle/working/corpus"],
}
subprocess.run(cmds[STAGE], check=True)

In [ ]:
# --- Publish -----------------------------------------------------------
# Outputs go to /kaggle/working (20 GB, auto-saved) and then become a Kaggle
# Dataset, which future GPU sessions mount instantly instead of downloading.
#
# One notebook output caps at 20 GB, so the 70 GB corpus is built by four
# sessions writing ~18 GB each and attached as four inputs.
from huggingface_hub import HfApi
import os

api = HfApi(token=os.environ["HF_TOKEN"])
api.upload_folder(
    repo_id="CHANGEME/bussin-corpus",
    repo_type="dataset",
    folder_path="/kaggle/working/corpus",
    path_in_repo=f"shards/part-{SHARD_INDEX:02d}",
)
print("pushed to the HF mirror; now Save Version to create the Kaggle Dataset")